# Check Phase 8 Checkpoint on HuggingFace
Download and inspect the checkpoint to verify `decoder_state_dict` is present.

In [ ]:
import os, sys
# Load token from .env file (never hardcode secrets)
env_path = os.path.join(os.getcwd(), '.env')
if os.path.exists(env_path):
    for line in open(env_path):
        if line.startswith('hf_token='):
            os.environ['HF_TOKEN'] = line.strip().split('=', 1)[1]
assert os.environ.get('HF_TOKEN'), 'Set HF_TOKEN env var or add hf_token= to .env'

from huggingface_hub import hf_hub_download

print('Downloading phase8.phase.pt from HuggingFace Hub...')
path = hf_hub_download(
    repo_id='UnseenGAP/FLUX',
    filename='checkpoints/phase8.phase.pt',
    token=os.environ['HF_TOKEN'],
)
size_mb = os.path.getsize(path) / 1e6
print(f'  Downloaded to: {path}')
print(f'  Size: {size_mb:.1f} MB')

In [ ]:
import torch

print('Loading checkpoint...')
ckpt = torch.load(path, map_location='cpu', weights_only=False)
print(f'Loaded.\n')

print('Top-level keys:')
for k in sorted(ckpt.keys()):
    v = ckpt[k]
    if isinstance(v, dict):
        print(f'  {k}: dict with {len(v)} entries')
    elif hasattr(v, 'shape'):
        print(f'  {k}: tensor {v.shape}')
    else:
        print(f'  {k}: {type(v).__name__} = {str(v)[:100]}')

In [ ]:
# ── Check decoder_state_dict ──
if 'decoder_state_dict' in ckpt:
    keys = list(ckpt['decoder_state_dict'].keys())
    has_prefix = any(k.startswith('_orig_mod.') for k in keys)
    print(f'✓ decoder_state_dict FOUND — {len(keys)} weight tensors')
    print(f'  Has _orig_mod. prefix (torch.compile): {has_prefix}')
    print()
    for k in keys:
        v = ckpt['decoder_state_dict'][k]
        print(f'  {k}: {v.shape} ({v.dtype})')
else:
    print('✗ decoder_state_dict NOT FOUND!')
    print()
    print('Available state_dict keys that contain "decoder" or "state":')
    for k in ckpt.keys():
        if 'decoder' in k.lower() or 'state' in k.lower():
            print(f'  {k}')

In [ ]:
# ── Metadata ──
print(f"phase: {ckpt.get('phase', '?')}")
print(f"timestamp: {ckpt.get('timestamp', '?')}")
print(f"learning_steps: {ckpt.get('learning_steps', '?')}")
print()

config = ckpt.get('config', {})
if config:
    print('Config:')
    for k in ['field_features', 'field_h', 'field_w', 'field_d', 'wave_dim']:
        print(f'  {k}: {config.get(k, "?")}')
print()

metrics = ckpt.get('metrics', {})
if metrics:
    print('Metrics:')
    for k, v in metrics.items():
        print(f'  {k}: {v}')